# Study on router

Use langchain to study router mode of agentic design.

In [1]:
import langchain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableBranch

print(f'Import LangChain V{langchain.__version__}')

Import LangChain V1.2.7


## Link to Model

In [2]:
# get api key from local file
with open('api-key/deepseek.txt', 'r') as f:
    api_key = f.read().strip()

In [3]:
# create LLM client
llm = ChatOpenAI(
    base_url="https://api.deepseek.com",
    api_key=api_key,
    model="deepseek-v4-flash",
    temperature=0,
)

## Demo from book

### Decision phase

Let LLM make decision according to user request.
And embed original request into output.

In [4]:
coordinator_router_prompt = ChatPromptTemplate.from_messages([
    ("system", """分析用户请求，判断应由哪个专属处理器处理。
     - 若请求涉及预订机票或酒店，输出 'booker'。
     - 其他一般信息问题，输出 'info'。
     - 若请求不明确或不属于上述类别，输出 'unclear'。
     只输出一个词：'booker'、'info' 或 'unclear'。"""),
    ("user", "{request}")
])

coordinator_router_chain = coordinator_router_prompt | llm | StrOutputParser()

decision_chain = {
    "decision": coordinator_router_chain,
    "origin": RunnablePassthrough()
}

### Branches

Define mock procedures of different decisions

In [5]:
booking_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是旅游公司负责机票和酒店预订的客服人员，根据用户请求，给出恰当的反馈"),
    ("user", "{origin}")
])

booking_handler = booking_prompt | llm | StrOutputParser()

In [6]:
info_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是旅游公司负责信息核对的客服人员，根据用户请求反馈合适的信息"),
    ("user", "{origin}")
])

info_handler = info_prompt | llm | StrOutputParser()

In [7]:
unclear_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是旅游公司的客服，用户输入的信息无法确认是预订请求还是信息咨询，你需要推测用户真实意图，并引导用户明确需求"),
    ("user", "{origin}")
])

unclear_handler = unclear_prompt | llm | StrOutputParser()

In [8]:
branches = RunnableBranch(
    (lambda x: x['decision'].strip() == 'booker', booking_handler),
    (lambda x: x['decision'].strip() == 'info', info_handler),
    unclear_handler
)

### Total proceed

In [9]:
coordinator_agent = decision_chain | branches

#### Testing

In [10]:
test_bench = [
    ("预订请求示例", "帮我预订飞往伦敦的机票。"),
    ("信息请求示例", "意大利的首都是哪里？"),
    ("不明请求示例", "哦啦啦啦啦～～～啦啦啦"),
]

In [11]:
for type, input in test_bench:
    print(f"--- {type} ---")
    print(f"用户请求：{input}")
    output = coordinator_agent.invoke({"request": input})
    print(f"智能体反馈：{output}")
    print()

--- 预订请求示例 ---
用户请求：帮我预订飞往伦敦的机票。
智能体反馈：您好！请问您希望从哪个城市出发？计划哪天出发，大概什么时间？另外，您需要预订几个人的机票？方便的话，请提供这些信息，我来为您查询合适的航班和价格。

--- 信息请求示例 ---
用户请求：意大利的首都是哪里？
智能体反馈：意大利的首都是罗马。

--- 不明请求示例 ---
用户请求：哦啦啦啦啦～～～啦啦啦
智能体反馈：您好！听起来您心情不错呢😊 请问您是想咨询旅游信息，还是有预订行程的需求呢？无论哪种，我都会尽力帮您～ 可以简单说说您想去哪里或者想了解什么吗？

